In [ ]:
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

train = pd.read_csv("../data/raw/train.csv")
test = pd.read_csv("../data/raw/test.csv")

feature_cols = [
    "CreditScore", "Geography", "Gender", "Age", "Tenure",
    "Balance", "NumOfProducts", "HasCrCard", "IsActiveMember", "EstimatedSalary",
]
cat_features = ["Geography", "Gender", "NumOfProducts", "HasCrCard", "IsActiveMember"]
y = train["Exited"].copy()

## Шаг 1-2. Почему бустинг, CatBoost и категориальные признаки

`NumOfProducts`, `HasCrCard`, `IsActiveMember` в файле хранятся как `float`
(1.0, 0.0 и т.п.), хотя по смыслу это категории/флаги. CatBoost с `float` в
`cat_features` упадёт с ошибкой, поэтому конвертируем в `int`.

**Грабли по дороге**: проверка `X[col].dtype != "object"` здесь не работает — в
этой версии pandas строковые колонки (`Geography`, `Gender`) имеют dtype `str`,
а не `object`, и такая проверка попыталась привести их к `int`
(`invalid literal for int() with base 10: 'France'`). Надёжнее проверять явно
числовой тип.

In [ ]:
def prepare(df, feature_cols, cat_features):
    X = df[feature_cols].copy()
    for col in cat_features:
        if pd.api.types.is_float_dtype(X[col]):
            X[col] = X[col].astype(int)
    return X

## Шаг 3-4. StratifiedKFold + CatBoost с early stopping

In [ ]:
def run_cv(X, y, cat_features, label):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    oof_preds = np.zeros(len(X))
    fold_scores = []
    models = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model = CatBoostClassifier(
            iterations=1000,
            learning_rate=0.05,
            depth=6,
            eval_metric="AUC",
            random_seed=42,
            verbose=False,
        )
        model.fit(
            X_tr, y_tr,
            cat_features=cat_features,
            eval_set=(X_val, y_val),
            early_stopping_rounds=50,
        )

        oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
        fold_auc = roc_auc_score(y_val, oof_preds[val_idx])
        fold_scores.append(fold_auc)
        models.append(model)
        print(f"[{label}] Fold {fold}: ROC AUC = {fold_auc:.4f}, best_iteration = {model.get_best_iteration()}")

    oof_auc = roc_auc_score(y, oof_preds)
    print(f"[{label}] OOF ROC AUC: {oof_auc:.4f}")
    print(f"[{label}] Mean fold ROC AUC: {np.mean(fold_scores):.4f} (+/- {np.std(fold_scores):.4f})")
    return oof_auc, models, fold_scores

In [ ]:
X_raw = prepare(train, feature_cols, cat_features)
oof_auc_raw, models_raw, scores_raw = run_cv(X_raw, y, cat_features, "raw")

[raw] Fold 0: ROC AUC = 0.9272, best_iteration = 87
[raw] Fold 1: ROC AUC = 0.9292, best_iteration = 91
[raw] Fold 2: ROC AUC = 0.9189, best_iteration = 119
[raw] Fold 3: ROC AUC = 0.9409, best_iteration = 221
[raw] Fold 4: ROC AUC = 0.9288, best_iteration = 127
[raw] OOF ROC AUC: 0.9289
[raw] Mean fold ROC AUC: 0.9290 (+/- 0.0070)


## Шаг 5. Сравнение с признаками туториала 4

In [ ]:
salary_cap = train["EstimatedSalary"].quantile(0.99)
bins = [0, 30, 40, 50, 60, 100]

train_fe = train.copy()
train_fe["EstimatedSalary"] = train_fe["EstimatedSalary"].clip(upper=salary_cap)
train_fe["AgeGroup"] = pd.cut(train_fe["Age"], bins=bins, labels=False)
train_fe["BalanceSalaryRatio"] = train_fe["Balance"] / (train_fe["EstimatedSalary"] + 1)

feature_cols_fe = feature_cols + ["AgeGroup", "BalanceSalaryRatio"]
cat_features_fe = cat_features + ["AgeGroup"]

X_fe = prepare(train_fe, feature_cols_fe, cat_features_fe)
oof_auc_fe, models_fe, scores_fe = run_cv(X_fe, y, cat_features_fe, "with tutorial-4 features")

[with tutorial-4 features] Fold 0: ROC AUC = 0.9266, best_iteration = 122
[with tutorial-4 features] Fold 1: ROC AUC = 0.9291, best_iteration = 121
[with tutorial-4 features] Fold 2: ROC AUC = 0.9182, best_iteration = 192
[with tutorial-4 features] Fold 3: ROC AUC = 0.9403, best_iteration = 137
[with tutorial-4 features] Fold 4: ROC AUC = 0.9289, best_iteration = 182
[with tutorial-4 features] OOF ROC AUC: 0.9285
[with tutorial-4 features] Mean fold ROC AUC: 0.9286 (+/- 0.0071)


**Результат сравнения**: сырые признаки — OOF 0.9289, признаки туториала 4 — OOF
0.9285. Разница в пределах шума, но сырые чуть лучше. Подтверждает гипотезу из
плана: `AgeGroup` и `BalanceSalaryRatio` были нужны линейной модели, а дерево
само находит похожие пороги по `Age`/`Balance` без подсказок — ручной feature
engineering для бустинга оказался не просто бесполезным, а слегка избыточным.
Берём **сырые признаки** для финальной модели.

## Шаг 6. Финальная модель и сабмит

In [ ]:
best_iterations = [m.get_best_iteration() for m in models_raw]
final_iterations = int(np.mean(best_iterations))
print(f"Best iterations per fold: {best_iterations}, using mean = {final_iterations}")

final_model = CatBoostClassifier(
    iterations=final_iterations,
    learning_rate=0.05,
    depth=6,
    random_seed=42,
    verbose=False,
)
final_model.fit(X_raw, y, cat_features=cat_features)

X_test = prepare(test, feature_cols, cat_features)
test_proba = final_model.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({"id": test["id"], "Exited": test_proba})
submission.to_csv("../submissions/catboost_cv.csv", index=False)

Best iterations per fold: [87, 91, 119, 221, 127], using mean = 129


## Итог туториала 5

```
Модель: CatBoostClassifier, сырые признаки, 5-fold StratifiedKFold
OOF ROC AUC: 0.9289 (было 0.9246 у логрегрессии с feature engineering)
Kaggle public score:  0.93935 (было 0.93238)
Kaggle private score: 0.93687 (было 0.92871)
```

**Цель проекта достигнута и перекрыта**: 0.93687 ставит модель примерно на
4-е место финального лидерборда (между текущими #3 — 0.93688 и #4 — 0.93666),
против изначальной цели "топ-10" (порог 0.93344).

Главный вывод этапа: смена модели с линейной на градиентный бустинг дала больший
эффект (+0.008 OOF, но резкий скачок в реальном Kaggle-score с 0.929 до 0.937),
чем весь ручной feature engineering туториала 4 вместе взятый, и без единого
вручную придуманного признака — бустинг сам нашёл нелинейности, которые в
туториале 4 пришлось искать через EDA и кодировать руками.